# EDA — Sensores IoT
**Jorge Andrés Mejía — 202300376**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
SENSORS = ['temperatura_c', 'presion_psi', 'vibracion_mm_s', 'potencia_kw']
SAVE  = True
IDIR  = 'imgs/'

def savefig(name):
    if SAVE:
        plt.savefig(f'{IDIR}{name}', dpi=150, bbox_inches='tight')

df = pd.read_csv('dataset_limpio.csv', parse_dates=['fecha_hora'])
print(f'Shape: {df.shape}')
df.head()

#### 1. Descripción general de los datos

In [ ]:
print('Tipos de datos:')
print(df.dtypes)
print(f'\nRegistros: {len(df)} | Columnas: {df.shape[1]}')
print(f'Máquinas únicas: {df["maquina_id"].nunique()}')
print(f'Rango temporal: {df["fecha_hora"].min()} → {df["fecha_hora"].max()}')
print(f'\nEstados operativos:\n{df["estado_operativo"].value_counts()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

counts = df['estado_operativo'].value_counts()
colors = {'OPERATIVO': '#70b0e0', 'MANTENIMIENTO': '#f0c060', 'FALLADO': '#e07070'}
axes[0].pie(counts.values, labels=counts.index, autopct='%1.0f%%',
            colors=[colors[k] for k in counts.index], startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[0].set_title('Distribución de estado operativo', fontweight='bold')

maq_counts = df['maquina_id'].value_counts().sort_index()
axes[1].bar(maq_counts.index, maq_counts.values, color='#70b0e0', edgecolor='white')
axes[1].set_title('Registros por máquina', fontweight='bold')
axes[1].set_xlabel('Máquina')
axes[1].set_ylabel('N° registros')

plt.tight_layout()
savefig('desc_general.png')
plt.show()

#### 2. Missing values, nulos y vacíos

In [ ]:
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Nulos': missing, 'Porcentaje (%)': pct})
print(missing_df)
print(f'\nTotal nulos en dataset limpio: {missing.sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.heatmap(df.isnull(), cbar=False, cmap='Reds', yticklabels=False, ax=axes[0])
axes[0].set_title('Mapa de nulos (dataset limpio)', fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=35, ha='right', fontsize=8)

cols_with_nulls = missing[missing > 0]
if len(cols_with_nulls):
    axes[1].barh(cols_with_nulls.index, cols_with_nulls.values, color='#e07070', edgecolor='white')
    axes[1].set_title('Columnas con nulos restantes', fontweight='bold')
    axes[1].set_xlabel('Cantidad')
else:
    axes[1].text(0.5, 0.5, 'Sin nulos', ha='center', va='center', fontsize=14)
    axes[1].set_title('Nulos restantes', fontweight='bold')
    axes[1].axis('off')

plt.tight_layout()
savefig('missing.png')
plt.show()

#### 3. Medidas de tendencia central

In [ ]:
stats = df[SENSORS].agg(['mean', 'median', 'std', 'min', 'max',
                          lambda x: x.skew(), lambda x: x.kurt()]).T
stats.columns = ['Media', 'Mediana', 'Desv. Estándar', 'Mínimo', 'Máximo', 'Asimetría', 'Curtosis']
stats = stats.round(3)
print(stats.to_string())
stats

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
colors = sns.color_palette('muted', 4)

for ax, col, color in zip(axes.flat, SENSORS, colors):
    mean   = df[col].mean()
    median = df[col].median()
    mode   = df[col].mode()[0]
    data   = df[col].dropna()

    sns.histplot(data, kde=True, ax=ax, color=color, bins=12, alpha=0.6)
    ax.axvline(mean,   color='red',    linestyle='--', lw=1.5, label=f'Media {mean:.1f}')
    ax.axvline(median, color='blue',   linestyle='-.',  lw=1.5, label=f'Mediana {median:.1f}')
    ax.axvline(mode,   color='green',  linestyle=':',   lw=1.5, label=f'Moda {mode:.1f}')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=8)

plt.suptitle('Medidas de tendencia central por variable', fontweight='bold')
plt.tight_layout()
savefig('tendencia_central.png')
plt.show()

#### 4. Histogramas y distribución de datos

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
colors = sns.color_palette('muted', 4)

for i, (col, color) in enumerate(zip(SENSORS, colors)):
    data = df[col].dropna()

    # Histograma
    axes[0][i].hist(data, bins=14, color=color, edgecolor='white', alpha=0.85)
    axes[0][i].set_title(f'Histograma\n{col}', fontweight='bold', fontsize=9)

    # Densidad
    sns.kdeplot(data, ax=axes[1][i], color=color, fill=True, alpha=0.4)
    axes[1][i].set_title(f'Densidad\n{col}', fontweight='bold', fontsize=9)

plt.suptitle('Histogramas y distribuciones de densidad', fontweight='bold')
plt.tight_layout()
savefig('histogramas.png')
plt.show()

#### 5. Evaluación de outliers — distribución y caja de bigotes

In [ ]:
# IQR method
for col in SENSORS:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    outliers = df[(df[col] < lo) | (df[col] > hi)]
    print(f'{col}: Q1={Q1:.2f}  Q3={Q3:.2f}  IQR={IQR:.2f}  límites=[{lo:.2f}, {hi:.2f}]  outliers={len(outliers)}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
colors = sns.color_palette('muted', 4)

for ax, col, color in zip(axes, SENSORS, colors):
    bp = ax.boxplot(df[col].dropna(), patch_artist=True, vert=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='o', color='red', markersize=5, alpha=0.6),
                    whiskerprops=dict(linewidth=1.2),
                    capprops=dict(linewidth=1.2))
    ax.set_title(col, fontweight='bold')
    ax.set_xticks([])

plt.suptitle('Caja de bigotes — dataset limpio (outliers IQR marcados en rojo)', fontweight='bold')
plt.tight_layout()
savefig('boxplots.png')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = sns.color_palette('muted', 4)

for ax, col, color in zip(axes, SENSORS, colors):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR

    sns.histplot(df[col], kde=True, ax=ax, color=color, bins=12, alpha=0.6)
    ax.axvline(lo, color='red', linestyle='--', lw=1.2, label='Límite IQR')
    ax.axvline(hi, color='red', linestyle='--', lw=1.2)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=8)

plt.suptitle('Distribución con límites IQR', fontweight='bold')
plt.tight_layout()
savefig('dist_iqr.png')
plt.show()

#### 6. Matriz y mapa de correlación

In [ ]:
corr = df[SENSORS].corr().round(3)
print('Matriz de correlación:')
print(corr)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap completo
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=axes[0], cbar_kws={'shrink': 0.8})
axes[0].set_title('Matriz de correlación (Pearson)', fontweight='bold')

# Triangular inferior
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=mask, square=True, linewidths=0.5, ax=axes[1], cbar_kws={'shrink': 0.8})
axes[1].set_title('Mapa de correlación (triángulo inferior)', fontweight='bold')

plt.tight_layout()
savefig('correlacion.png')
plt.show()

In [ ]:
# Pairplot para ver relaciones entre todas las variables
g = sns.pairplot(df[SENSORS + ['estado_operativo']], hue='estado_operativo',
                 palette={'OPERATIVO': '#70b0e0', 'MANTENIMIENTO': '#f0c060', 'FALLADO': '#e07070'},
                 plot_kws=dict(alpha=0.7, s=40), diag_kind='kde')
g.figure.suptitle('Pairplot por estado operativo', y=1.01, fontweight='bold')
savefig('pairplot.png')
plt.show()

#### 7. Visualizaciones adicionales para el reporte

In [ ]:
# Boxplots de sensores por estado operativo
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
palette = {'OPERATIVO': '#70b0e0', 'MANTENIMIENTO': '#f0c060', 'FALLADO': '#e07070'}
order   = ['OPERATIVO', 'MANTENIMIENTO', 'FALLADO']

for ax, col in zip(axes, SENSORS):
    sns.boxplot(data=df, x='estado_operativo', y=col, order=order,
                palette=palette, ax=ax, linewidth=1.2)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticklabels(order, rotation=20, ha='right', fontsize=8)

plt.suptitle('Sensores por estado operativo', fontweight='bold')
plt.tight_layout()
savefig('sensores_estado.png')
plt.show()

In [ ]:
# Temperatura promedio y tasa de fallos por máquina
summary = df.groupby('maquina_id').agg(
    temp_media   = ('temperatura_c', 'mean'),
    vibracion_media = ('vibracion_mm_s', 'mean'),
    tasa_fallos  = ('estado_operativo', lambda x: (x == 'FALLADO').mean() * 100)
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(summary['maquina_id'], summary['temp_media'], color='#70b0e0', edgecolor='white')
axes[0].axhline(summary['temp_media'].mean(), color='gray', linestyle='--', lw=1.2)
axes[0].set_title('Temperatura media por máquina', fontweight='bold')
axes[0].set_ylabel('°C')
axes[0].set_xticklabels(summary['maquina_id'], rotation=45)

axes[1].bar(summary['maquina_id'], summary['vibracion_media'], color='#90c97a', edgecolor='white')
axes[1].axhline(summary['vibracion_media'].mean(), color='gray', linestyle='--', lw=1.2)
axes[1].set_title('Vibración media por máquina', fontweight='bold')
axes[1].set_ylabel('mm/s')
axes[1].set_xticklabels(summary['maquina_id'], rotation=45)

bar_colors = ['#e07070' if v > 0 else '#70b0e0' for v in summary['tasa_fallos']]
axes[2].bar(summary['maquina_id'], summary['tasa_fallos'], color=bar_colors, edgecolor='white')
axes[2].set_title('Tasa de fallos por máquina (%)', fontweight='bold')
axes[2].set_ylabel('%')
axes[2].set_xticklabels(summary['maquina_id'], rotation=45)

plt.tight_layout()
savefig('resumen_maquinas.png')
plt.show()

In [ ]:
print('Imágenes guardadas en imgs/')
import os
for f in sorted(os.listdir('imgs/')):
    print(' ', f)